# Visualization part 2

The objective of this tutorial is to:
- Create your first widget
- Add interactivity elements to a Plotly chart
- transform the interactive visualization notebook into a webapp (web application) with Voilà
- deploy and share this web application with ngrok

# 1) Creation of a first widget

a) Installation of the Pywi library

In [ ]:
import sys
!{sys.executable} -m --quiet pip install ipywidgets statsmodels

import os
shared_path = os.path.abspath(os.path.join(os.getcwd(), "../Shared"))
if shared_path not in sys.path:
    sys.path.append(shared_path)
from datetime import datetime

/opt/anaconda3/envs/TheWagon/bin/python: No module named --quiet


b) Run the code below to import the library and rename it widgets. We will use

In [2]:
import ipywidgets as widgets
from IPython.display import display

c) The full list of available widgets is available at [this address](https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20List.html).

Using this documentation, construct a variable `my_int_slider` which will contain an "IntSlider" widget with 5 as default value, 1 as minimum and 20 as maximum. We can leave step equal to 1 as the increment level.

Use the "display" function to display it.

In [3]:
my_int_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
)

display(my_int_slider)

IntSlider(value=5, max=20, min=1)

d) What is the result of this operation?

In [4]:
my_int_slider.keys

['_dom_classes',
 '_model_module',
 '_model_module_version',
 '_model_name',
 '_view_count',
 '_view_module',
 '_view_module_version',
 '_view_name',
 'behavior',
 'continuous_update',
 'description',
 'description_allow_html',
 'disabled',
 'layout',
 'max',
 'min',
 'orientation',
 'readout',
 'readout_format',
 'step',
 'style',
 'tabbable',
 'tooltip',
 'value']

e) Deduce how to display the value of the slider.

Vary the value of the slider and check that by executing the line of code again, the value has changed.

In [5]:
my_int_slider.value

5

# 2) Creation of a User Form

Widgets can be used to create user forms for end users.

The purpose of this exercise is to build a tool that allows you to select a stock market index to display a scatter plot with the Apple AAPL index.

a) Import the following libraries

In [6]:
import pandas as pd
import plotly.express as px
from ipywidgets import interact, interactive, fixed, interactive_output
import io

b) Read this file [csv example](https://drive.google.com/file/d/1bWMYHBP6pnxB-sztTyRe8gGMszD0GOp4/view?usp=sharing) into a Pandas DataFrame.

In [7]:
df = pd.read_csv("./stock_data.csv")
# Ensure date is considered as an actual date
df.loc[:, "date"] = pd.to_datetime(df.date)
display(df.head(5))

/var/folders/yb/fs1cv8ls291dcbt3p83rmkw40000gn/T/ipykernel_11837/889337687.py:3: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[:, "date"] = pd.to_datetime(df.date)


,date,GOOG,AAPL,AMZN,FB,NFLX,MSFT
0,2018-01-01,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,2018-01-08,1.018172,1.011943,1.061881,0.959968,1.053526,1.015988
2,2018-01-15,1.032008,1.019771,1.053240,0.970243,1.049860,1.020524
3,2018-01-22,1.066783,0.980057,1.140676,1.016858,1.307681,1.066561
4,2018-01-29,1.008773,0.917143,1.163374,1.018357,1.273537,1.040708


c) Represent in a Plotly Scatter graph the data of GOOG versus APPL with a trendline (trendlines='ols') to see the correlation between these two indices.

In [8]:
labels_dict = {
    "date" : "Date",
    "value" : "Share Price Evolution",
    "variable" : "Company"
}

fig = px.scatter(
    df,
    x="GOOG",
    y="AAPL",
    trendline="ols",
    title="Share Price Performance Comparison",
    labels=labels_dict
)

# Rename entries into the legend
fig.for_each_trace(lambda title: title.update(name=title.name.replace("GOOG", "Google").replace("AAPL", "Apple")))

fig.show()

d) Create a `financial_scatter_widget` function which will take as input the parameters df, x and y, it will display the graph created earlier for the column names x and y.

In [ ]:
def financial_scatter_widget(df, x=None, y=None):
    fig = px.scatter(
        df,
        x=x,
        y=y,
        trendline="ols",
        title="Share Price Performance Comparison",
        labels=labels_dict,
    )

    fig.show()


financial_scatter_widget(df, x="GOOG", y="AAPL")

e) The idea is to give the hand to the user to choose which index he wishes to verify the correlation with the APPL index.

Create a Dropdown type widget, which will select the name of a column of the DataFrame created.

In [10]:
x_dropdown = widgets.Dropdown(
    options=["GOOG", "MSFT", "AMZN", "FB", "NFLX"],
    value="GOOG",
    description="AAPL vs ",
    style={"description_width": "initial"},
    disabled=False,
)

display(x_dropdown)

Dropdown(description='AAPL vs ', options=('GOOG', 'MSFT', 'AMZN', 'FB', 'NFLX'), style=DescriptionStyle(descri…

f) Using the widgets `interact()` function, execute the `financial_scatter_widget` function depending on the chosen column.

We will set the parameters `df`=The DataFrame retrieved from the upload and `y`="APPL" using the `fixed()` function.

In [11]:
widgets.interact(
    financial_scatter_widget,
    df = fixed(df),
    x = x_dropdown,
    y = fixed("AAPL"),
)

interactive(children=(Dropdown(description='AAPL vs ', options=('GOOG', 'MSFT', 'AMZN', 'FB', 'NFLX'), style=D…

<function __main__.financial_scatter_widget(df, x=None, y=None)>

# 3) Realization of the Restorative Dashboard

We are going to take the visualizations made in the last lab and add two interaction elements, the restaurant ID and the dates of the selected period.

a) We are going to take the data from the previous lab. If needed, you can download them again at [this address](https://drive.google.com/file/d/1tSA-l-ziWLk90iX48eHne_8aPfx8braI/view?usp=sharing).

b) Run the following data preprocessing code.

In [12]:
df = pd.read_csv("/Users/gachdel/anaconda_projects/2_Fundamentals_Data_Structure/Tiller_order_data.csv")

# Transforms dates into date format
df["date_opened"] = pd.to_datetime(df["date_opened"])
df["date_closed"] = pd.to_datetime(df["date_closed"])

# Index the order opening date
df = df.set_index(df["date_opened"])

/var/folders/yb/fs1cv8ls291dcbt3p83rmkw40000gn/T/ipykernel_11837/2890174931.py:1: DtypeWarning:

Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.



In [13]:
df

,id_order,id_store,id_table,id_waiter,id_customer,id_external,id_device,date_opened,date_closed,dim_name,dim_status,dim_type,dim_comment,dim_source,m_nb_customer,m_cached_payed,m_cached_price
date_opened,,,,,,,,,,,,,,,,,
2019-01-12 13:02:17+00:00,55538867,8052,NaN,NaN,NaN,0425716B-EFF4-41CA-AEA1-839104F36833,15327.0,2019-01-12 13:02:17+00:00,2019-01-12 19:58:38+00:00,vincent,CLOSED,1,NaN,Tiller iPAD,1,45.5,45.5
2019-01-16 19:39:09+00:00,56035309,8052,NaN,NaN,NaN,75E41FE2-64FF-41D3-954C-A7DE4AA887EF,15327.0,2019-01-16 19:39:09+00:00,2019-01-16 22:10:50+00:00,frere et soeur avec pierre,CLOSED,1,NaN,Tiller iPAD,2,49.8,49.8
2019-01-12 14:18:46+00:00,55550051,8052,NaN,NaN,NaN,F6051A05-C9AC-4033-BF72-BB5149B8F439,15327.0,2019-01-12 14:18:46+00:00,2019-01-12 19:50:32+00:00,rachel,CLOSED,1,NaN,Tiller iPAD,1,27.4,27.4
2019-01-24 17:49:12+00:00,57000119,8052,NaN,16199.0,NaN,B8BEEC66-1C10-48A0-B4D5-035CB5EEFE62,15327.0,2019-01-24 17:49:12+00:00,2019-01-24 21:58:59+00:00,Groupe PEL,CLOSED,1,NaN,tiller-order,3,60.0,60.0
2019-01-12 15:25:06+00:00,55558817,8052,NaN,NaN,NaN,17F0533C-2FF1-4FC5-A50D-12704C7B7A4B,15327.0,2019-01-12 15:25:06+00:00,2019-01-12 19:21:03+00:00,remi et date,CLOSED,1,NaN,Tiller iPAD,2,39.9,39.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-02-28 13:13:21+00:00,61630471,4542,NaN,NaN,NaN,C4A0AFD8-FBD7-4740-A48B-4890F46E43E2,7411.0,2019-02-28 13:13:21+00:00,2019-02-28 13:38:45+00:00,NaN,CLOSED,1,NaN,Tiller iPAD,1,23.0,23.0
2019-06-11 11:08:01+00:00,76772730,4542,NaN,NaN,NaN,95A17B0B-22AE-43A1-8BEB-B105BF337655,7411.0,2019-06-11 11:08:01+00:00,2019-06-11 11:10:26+00:00,NaN,CLOSED,1,NaN,Tiller iPAD,1,26.0,26.0
2019-06-17 10:30:50+00:00,77799227,4542,NaN,NaN,NaN,0E5D4843-7EA3-4D9E-9490-EB9717A0220F,7411.0,2019-06-17 10:30:50+00:00,2019-06-17 10:32:45+00:00,NaN,CLOSED,1,NaN,Tiller iPAD,1,29.0,29.0


c) Run the code below which determines the restaurant id and the dates of the desired period. These are the parameters that we are going to make selectable using widgets.

In [14]:
id_store = 8052
date_debut = "2019-10-01"
date_fin = "2021-12-31"

df_store = df[
    (df.id_store == id_store)
    & (df.date_opened >= date_debut)
    & (df.date_closed < date_fin)
]
df_store.shape

(2703, 17)

d) Change the previous code using two widgets, a dropdown menu to choose the restaurant and a date picker to choose the start and end date of the period you want to analyze.

Create the `id_store_dropdown` variable with a Dropdown type widget:

In [15]:
id_store_dropdown = widgets.Dropdown(
    options=df.id_store.unique(),
    value=df.id_store.unique()[0],
    description="Restaurant: ",
    style={"description_width": "initial"},
)

Create the variable `date_start_picker` with a widget of type DatePicker:

In [16]:
""" CHALLENGE: when the store is updated using the dropdown, update date_debut_picker and date_fin_picker"""

# Get the date_opened for the currently selected restaurant
df_date_opened=df.date_opened[df.id_store == id_store_dropdown.value]

date_debut_picker = widgets.DatePicker(
    value=df_date_opened[0],
    description="Start Date: ",
    style={"description_width": "initial"},
)

Create the variable `date_end_picker` with a widget of type DatePicker:

In [17]:
""" CHALLENGE: when the store is updated using the dropdown, update date_debut_picker and date_fin_picker"""

# Get the open date_closed for the currently selected restaurant
df_date_closed = df.date_closed[df.id_store == id_store_dropdown.value]

date_fin_picker = widgets.DatePicker(
    options=df_date_closed,
    value=df_date_closed.unique()[-1],
    description="End Date: ",
    style={"description_width": "initial"},
)

e) Run the following code to transform all the visualizations made in the previous lab into functions

In [18]:
import plotly.graph_objects as go
""" CHALLENGE: review revenue_over_time x axis scale """
""" CHALLENGE: display an error message when the dataframe is empty """


def revenue_over_time(df_store):
    print(len(df_store))
    df_revenue = df_store[df_store.dim_status=="CLOSED"].resample("D").m_cached_payed.sum()

    fig = px.line(df_revenue,
                  y="m_cached_payed",
                  title="Revenue over time",
                  range_x=[df_revenue.index.min(), df_revenue.index.max()],
                  range_y=[df_revenue.values.min() / 1.1, df_revenue.values.max() * 1.1],)
    fig.update_xaxes(
        rangeselector=dict(
            buttons=list([
                dict(count=7, label="Last 7days", step="day", stepmode="backward"),
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        )
    )
    fig.show()

def clients_over_time(df_store):
    df_revenue = df_store[df_store.dim_status=="CLOSED"].resample("D").m_cached_payed.sum()
    df_nb_customer = df_store[df_store.dim_status=="CLOSED"].resample("D")["m_nb_customer"].sum()

    df_average_basket = df_revenue / df_nb_customer

    fig_panier_moyen = go.Scatter(x=df_average_basket.index,
                                  y=df_average_basket,
                                  name="Avg basket",
                                  mode="lines",
                                  line_color="#000000")

    fig = px.bar(df_nb_customer,
                 title="Customers through time")
    fig.add_trace(fig_panier_moyen)
    fig.update_xaxes(tickformat="%d/%m/%Y")
    fig.update_xaxes(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1m", step="month", stepmode="backward"),
                dict(count=6, label="6m", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        )
    )
    fig.show()

def best_staff(df_store):
    df_store.loc[df_store["id_waiter"].isnull(), "id_waiter"] = "Unknown"

    df_waiter = df_store.groupby("id_waiter", as_index=False)["m_cached_payed"].sum()

    fig = px.pie(
        df_waiter,
        names='id_waiter',
        values='m_cached_payed',
        color='id_waiter',
        title = 'Split of sales by waiter'
    )
    fig.update_traces(
        textinfo='percent+label'
    )
    fig.show()

f) Make the previously created widgets interact, via the 'interact' function, with the 'display_dash' function.

This 'display_dash' function filters the initial dataframe based on the values ​​specified in the three widgets, and then calls the visualizations above to display them from the filtered dataframe.

You must therefore, in this 'display_dash' function:
- filter the initial dataframe (df) based on the values ​​of the id_store_dropdown, date_start_picker and date_end_picker widgets
- call the functions revenue_over_time, clients_over_time and best_staff

In [19]:
def display_dash(df, id_store_dropdown, date_debut_picker, date_fin_picker):
    # Convert picker values into pandas.Timestamp (to compare to pandas.datetime)
    date_debut_picker = pd.Timestamp(date_debut_picker).date()
    date_fin_picker = pd.Timestamp(date_fin_picker).date()
    mask = (
        (df.id_store == id_store_dropdown)
        & (df.date_opened.dt.date >= date_debut_picker)
        & (df.date_closed.dt.date <= date_fin_picker)
    )
    df_filtered = df.loc[mask, :]
    # Calling the revenue_over_time, clients_over_time and best_staff functions :
    revenue_over_time(df_filtered)
    clients_over_time(df_filtered)
    best_staff(df_filtered)

In [20]:
out = widgets.interact(
    display_dash,
    df=fixed(df),
    id_store_dropdown=id_store_dropdown,
    date_debut_picker=date_debut_picker,
    date_fin_picker=date_fin_picker,
)

display(out)

interactive(children=(Dropdown(description='Restaurant: ', options=(8052, 8347, 8283, 9084, 4337, 360, 1796, 7…

<function __main__.display_dash(df, id_store_dropdown, date_debut_picker, date_fin_picker)>

g) Organize your dashboard using HBox and VBox from the widgets library.

They are used to manage the layout of the different visualizations:
- place the widget on the first line: id_store_dropdown
- place the date_start_picker and date_end_picker widgets on the second line

In [21]:
from ipywidgets import HBox, VBox

In [22]:
first_line = HBox([id_store_dropdown])
second_line = HBox([date_debut_picker, date_fin_picker])

ui = VBox([first_line, second_line])

In [23]:
ui

h) now use the function ['interactive_output'](https://ipywidgets.readthedocs.io/en/latest/examples/Using%20Interact.html#More-control-over-the-user-interface: -interactive_output).
This function allows you to be more flexible on the arrangement of widgets.

In [24]:
out = interactive_output(
    display_dash,
    {
        "df": fixed(df),
        "id_store_dropdown": id_store_dropdown,
        "date_debut_picker": date_debut_picker,
        "date_fin_picker": date_fin_picker,
    },
)
display(ui, out)

Output()

i) Check that everything is working fine by changing the restaurant id and period dates.
If everything works, congratulations you have created your first dashboard in Python!

# 4) Transformation of the notebook into a web application with Voilà

Now that we have all the elements for our user dashboard, we are going to convert our notebook into a web application so that we can run it locally and be able to interact with the elements.

a) Start by installing [Voila](https://github.com/voila-dashboards/voila)

In [25]:
import sys
!{sys.executable} -m pip install voila

b) Mark all the cells to keep only the cells of exercise 3, in order to display only the dashboard of this part.

To do this, you will first need to tag the cells with a tag name. We will choose the `hide` tag in this example.

To do this, click on "View"-> "Cell Toolbar" -> "Tags"

(Ah... wait, it's already done! We save you time)

c) From a terminal, run the command:
```
voila --TagRemovePreprocessor.remove_cell_tags hide ce_notebook.ipynb
```

If you have any issues rendering in your Voila notebook, running `pip install --upgrade ipywidgets` should sort them out for you!\\

# 5) Deploying the webapp with ngrok

We now want to be able to share our webapp without deploying it on a server on another computer.

For this, we will use the ngrok library, the documentation of which can be consulted at [this address](https://voila.readthedocs.io/en/stable/deploy.html). But we will guide you in the rest of this part.

a) Install [ngrok](https://ngrok.com/download)

https://ngrok.com/download

b) Unzip the archive, then authenticate with your token in the CLI (you will need to already have a token - on the download page you will see a sign up link for a token here https://dashboard.ngrok.com/signup):

```./ngrok authtoken <token>```
OR (depending on your installation process.
```ngrok authtoken <token>```

c) Make sure that a webapp voilà is already launched locally (see command voila above)

d) Launch a tunnel on port 8866:

```ngrok http 8866``` or ```./ngrok http 8866```

e) You should see something like this:
<img src = "https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/Transformers/Screenshot%202023-07-21%20at%2010.35.13%20AM.png">
Click on the link displayed in the terminal (```Forwarding```).
You can send this url to your neighbour, they will be able to access your webapp!

# [Bonus] Deployment of a webapp with Binder

To be able to use [Binder](https://jupyter.org/binder), you must first have pushed your notebook into a git repo and added a 'requirements.txt' file containing the libraries necessary to run your project.

Then just go to https://mybinder.org/ and:
- in 'GitHub', copy paste the URL of your git project
- in 'Path to a notebook file (optional)', write 'voila/render/ce_notebook.ipynb' specifying 'URL' instead of 'File'. This allows you to specify to Binder that you must directly use Voila to create the webapp.

Then click on 'launch'. A docker image of your project will be created using the 'requirements.txt' file. You will then have a link to launch a standalone docker container that will allow you to access your web application. For each user using this link, a docker container will be created.